In [1]:
%run ../../Utils/yp_utils.py

# Initial setup

In [2]:
paper_pmid = 36738789
paper_name = 'shimasawa_mizushima_2023' 

In [3]:
datasets = pd.read_csv('extras/YeastPhenome_' + str(paper_pmid) + '_datasets_list.txt', sep='\t', header=None, names=['dataset_id', 'name'])

In [4]:
datasets.set_index('dataset_id', inplace=True)

# Load & process the data

In [11]:
original_data = pd.read_excel('raw_data/mmc1.xlsx', header=3, sheet_name='Sheet1')

In [12]:
print('Original data dimensions: %d x %d' % (original_data.shape))

Original data dimensions: 4701 x 16


In [13]:
original_data.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Normalized Mean Electronic Volume [A.U.],Median volume [fL],Normalized,Mean Volume [fL],Normalized .1,Area of Mother Cell [pixel2],Normalized .2,Median FSC,Normalized .3,Mean Volume [fL].1,Normalized .4,Median FSC.1,Normalized .5
0,MatA_17_h8,YKL055C,OAR1,8.594,38.792353,0.946,39.57,0.680,890.932584,0.988,86.396667,0.939,NaN,NaN,38119.146272,0.976
1,MatA_17_b10,YLR182W,SWI6,6.886,NaN,NaN,NaN,NaN,1118.751351,1.241,NaN,NaN,NaN,NaN,NaN,NaN
2,MatA_70_a3,YLR337C,VRP1,6.512,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MatA_19_a11,YGR036C,CAX4,6.304,NaN,NaN,65.49,1.125,986.952381,1.095,NaN,NaN,NaN,NaN,NaN,NaN
4,MatA_27_g2,YBL094C,NaN,6.165,NaN,NaN,51.33,0.882,1556.142857,1.726,96.210000,1.046,NaN,NaN,NaN,NaN


In [14]:
original_data['orf'] = original_data['Unnamed: 1'].astype(str)

In [15]:
# Eliminate all white spaces & capitalize
original_data['orf'] = clean_orf(original_data['orf'])

In [16]:
# Translate to ORFs 
original_data['orf'] = translate_sc(original_data['orf'], to='orf')

In [17]:
# Make sure everything translated ok
t = looks_like_orf(original_data['orf'])
print(original_data.loc[~t,])

Empty DataFrame
Columns: [Unnamed: 0, Unnamed: 1, Unnamed: 2, Normalized Mean Electronic Volume [A.U.], Median volume [fL], Normalized , Mean Volume [fL], Normalized .1, Area of Mother Cell [pixel2], Normalized .2, Median FSC, Normalized .3, Mean Volume [fL].1, Normalized .4, Median FSC.1, Normalized .5, orf]
Index: []


In [18]:
original_data['data'] = original_data['Normalized Mean Electronic Volume [A.U.]']

In [19]:
original_data.set_index('orf', inplace=True)

In [20]:
original_data = original_data[['data']].copy()

In [21]:
original_data = original_data.groupby(original_data.index).mean()

In [22]:
original_data.shape

(4645, 1)

# Prepare the final dataset

In [23]:
data = original_data.copy()

In [24]:
dataset_ids = [22285]
datasets = datasets.reindex(index=dataset_ids)

In [25]:
lst = [datasets.index.values, ['value']*datasets.shape[0]]
tuples = list(zip(*lst))
idx = pd.MultiIndex.from_tuples(tuples, names=['dataset_id','data_type'])
data.columns = idx

In [26]:
data.head()

dataset_id,22285
data_type,value
orf,
YAL002W,1.003
YAL004W,1.085
YAL005C,0.819
YAL007C,1.253
YAL008W,1.072


## Subset to the genes currently in SGD

In [27]:
genes = pd.read_csv(path_to_genes, sep='\t', index_col='id')
genes = genes.reset_index().set_index('systematic_name')
gene_ids = genes.reindex(index=data.index.values)['id'].values
num_missing = np.sum(np.isnan(gene_ids))
print('ORFs missing from SGD: %d' % num_missing)

ORFs missing from SGD: 15


In [28]:
data['gene_id'] = gene_ids
data = data.loc[data['gene_id'].notnull()]
data['gene_id'] = data['gene_id'].astype(int)
data = data.reset_index().set_index(['gene_id','orf'])

data.head()

,dataset_id,22285
,data_type,value
gene_id,orf,
2,YAL002W,1.003
1863,YAL004W,1.085
4,YAL005C,0.819
5,YAL007C,1.253
6,YAL008W,1.072


# Normalize

In [29]:
data_norm = normalize_phenotypic_scores(data, has_tested=True)

In [30]:
# Assign proper column names
lst = [datasets.index.values, ['valuez']*datasets.shape[0]]
tuples = list(zip(*lst))
idx = pd.MultiIndex.from_tuples(tuples, names=['dataset_id','data_type'])
data_norm.columns = idx

In [31]:
data_norm[data.isnull()] = np.nan
data_all = data.join(data_norm)

data_all.head()

dataset_id       22285          
data_type        value    valuez
gene_id orf                     
2       YAL002W  1.003 -0.067694
1863    YAL004W  1.085  0.138125
4       YAL005C  0.819 -0.529531
5       YAL007C  1.253  0.559803
6       YAL008W  1.072  0.105495

# Print out

In [32]:
for f in ['value','valuez']:
    df = data_all.xs(f, level='data_type', axis=1).copy()
    df.columns = datasets['name'].values
    df = df.droplevel('gene_id', axis=0)
    df.to_csv(paper_name + '_' + f + '.txt', sep='\t')

# Save to DB

In [35]:
import sys
sys.path.append('/Users/anastasia/Lab/Utils/Python/')

In [36]:
from IO.save_data_to_db3 import *

In [37]:
save_data_to_db(data_all, paper_pmid)

Deleting all datasets for PMID 36738789...
Inserting the new data...
22285
Updating the data_modified_on field...
